## Problem Statement

### Context

Wine is produced by the fermentation of grapes and other fruit juices and ranks as the second most popular alcoholic beverage worldwide after beer. Traditionally, wine quality is assessed by expert tasters who evaluate flavour, aroma and vintage. This sensory process is time-consuming, expensive and subjective.

Quality is also strongly influenced by measurable physiochemical properties such as fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, sulphur dioxide levels, density, pH, sulphates and alcohol content.

### Objective

Moonshine is a premium red-wine producer that wishes to raise production efficiency and cut the cost and delay associated with manual tasting. As the data scientist at Moonshine you are required to:

- Identify the key physiochemical factors that distinguish high-quality (premium) wines.
- Build a reliable predictive model that can flag premium wines automatically.
- Select an appropriate evaluation metric given the class imbalance typical of quality scores.
- Provide actionable recommendations that help the company prioritise batches and optimise quality-control resources.

### Data Description

| Feature | Description |
|---------|-------------|
| fixed acidity | Grams of tartaric acid per dm³ |
| volatile acidity | Grams of acetic acid per dm³ |
| citric acid | Grams of citric acid per dm³ |
| residual sugar | Grams of sugar remaining after fermentation per dm³ |
| chlorides | Grams of sodium chloride per dm³ |
| free sulfur dioxide | Grams of free sulfite per dm³ |
| total sulfur dioxide | Grams of total sulfite per dm³ |
| density | Density in g/cm³ |
| pH | Acidity/basicity scale (0–14) |
| sulphates | Grams of potassium sulphate per dm³ |
| alcohol | Alcohol content by volume (%) |
| quality | Sensory score from 3 to 8 (target) |

## Importing Necessary Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Oversampling and undersampling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.3f" % x)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Loading the Dataset

In [ ]:
data = pd.read_csv("winequality.csv", sep=";")
print("Shape of the dataset:", data.shape)
data.head()

## Data Preprocessing

In [ ]:
# Create a working copy
df = data.copy()

# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())

## Exploratory Data Analysis (EDA)

### Univariate Analysis – Target Variable (quality)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="quality", data=df, ax=ax[0], palette="viridis")
ax[0].set_title("Distribution of Wine Quality Scores")

sns.boxplot(x=df["quality"], ax=ax[1], color="lightcoral")
ax[1].set_title("Boxplot of Wine Quality")

plt.tight_layout()
plt.show()

print(df["quality"].value_counts().sort_index())
print(f"\nMean quality: {df['quality'].mean():.2f}")

**Observations:**
- Quality scores range from 3 to 8; majority are rated 5 or 6.
- Premium scores (7 and 8) form a minority class (~13.6%).
- Class imbalance must be considered.

### Defining the binary target – Premium vs Non-Premium

In [ ]:
# Premium = quality >= 7
df["premium"] = (df["quality"] >= 7).astype(int)

print("Premium class distribution:")
print(df["premium"].value_counts())
print(f"\nPremium proportion: {df['premium'].mean()*100:.1f}%")

### Univariate Analysis – Continuous Features

In [ ]:
num_cols = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide",
    "density", "pH", "sulphates", "alcohol"
]

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribution of {col}")

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

**Observations:**
- Most features are right-skewed (residual sugar, chlorides, SO₂).
- Alcohol and pH are closer to normal.
- Outliers exist; tree-based models are robust to them.

### Bivariate Analysis – Features vs Premium Class

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x="premium", y=col, data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by Premium Status")
    axes[i].set_xticklabels(["Non-Premium", "Premium"])

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

**Observations:**
- Alcohol is markedly higher in premium wines.
- Volatile acidity is lower in premium wines.
- Citric acid and sulphates tend to be higher in premium wines.

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    df[num_cols + ["quality", "premium"]].corr(),
    annot=True, fmt=".2f", cmap="Spectral", center=0, linewidths=0.5
)
plt.title("Correlation Matrix")
plt.show()

**Observations:**
- Alcohol has the strongest positive correlation with quality/premium.
- Volatile acidity has a clear negative correlation.
- Free and total sulfur dioxide are highly correlated with each other.

## EDA Summary

- Premium wines (score ≥ 7) represent only ~13.6% of samples → class imbalance.
- Alcohol is the strongest separator.
- Lower volatile acidity, higher citric acid and sulphates are associated with better scores.
- Tree-based ensembles are suitable because of skewness and outliers.

## Feature Engineering

In [ ]:
# Simple ratio feature
df["free_total_so2_ratio"] = df["free sulfur dioxide"] / (df["total sulfur dioxide"] + 1e-5)

# Acidity difference
df["acidity_diff"] = df["fixed acidity"] - df["volatile acidity"]

print("Columns after feature engineering:")
print(df.columns.tolist())
print("Shape:", df.shape)

## Data Preparation for Modelling

In [ ]:
X = df.drop(columns=["quality", "premium"])
y = df["premium"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Training set:", X_train.shape[0])
print("Test set    :", X_test.shape[0])
print("\nClass balance – Train:")
print(y_train.value_counts(normalize=True))

## Handling Class Imbalance – Oversampling & Undersampling

In [ ]:
# SMOTE (Oversampling)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_smote).value_counts())

In [ ]:
# Random Under-sampling
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("After RandomUnderSampler:")
print(pd.Series(y_train_rus).value_counts())

**Note:** We will train the main models on the SMOTE-balanced training set. Test set remains unchanged.

## Model Building – Ensemble Techniques

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_test, y_test, model_name="Model"):
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    
    if hasattr(model, "predict_proba"):
        y_prob_test = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob_test)
    else:
        roc_auc = np.nan
    
    metrics_dict = {
        "Model"    : model_name,
        "Train Acc": metrics.accuracy_score(y_train, y_pred_train),
        "Test Acc" : metrics.accuracy_score(y_test, y_pred_test),
        "Precision": metrics.precision_score(y_test, y_pred_test),
        "Recall"   : metrics.recall_score(y_test, y_pred_test),
        "F1-Score" : metrics.f1_score(y_test, y_pred_test),
        "ROC-AUC"  : roc_auc
    }
    return metrics_dict, model

### 1. Decision Tree (Baseline) – SMOTE

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_metrics, dt_fitted = evaluate_classifier(
    dt_model, X_train_smote, y_train_smote, X_test, y_test, "Decision Tree (SMOTE)"
)
pd.DataFrame([dt_metrics])

### 2. Bagging – SMOTE

In [ ]:
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
bag_metrics, bag_fitted = evaluate_classifier(
    bag_model, X_train_smote, y_train_smote, X_test, y_test, "Bagging (SMOTE)"
)
pd.DataFrame([bag_metrics])

### 3. Random Forest – Hyperparameter Tuning + SMOTE

In [ ]:
rf_params = {
    "n_estimators": [100, 150],
    "max_depth": [10, 12],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    param_grid=rf_params,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

rf_grid.fit(X_train_smote, y_train_smote)
print("Best parameters:", rf_grid.best_params_)

rf_metrics, rf_fitted = evaluate_classifier(
    rf_grid.best_estimator_, X_train_smote, y_train_smote, X_test, y_test, "Random Forest (Tuned + SMOTE)"
)
pd.DataFrame([rf_metrics])

### 4. AdaBoost – SMOTE

In [ ]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"),
    n_estimators=50,
    learning_rate=0.1,
    random_state=42
)
ada_metrics, ada_fitted = evaluate_classifier(
    ada_model, X_train_smote, y_train_smote, X_test, y_test, "AdaBoost (SMOTE)"
)
pd.DataFrame([ada_metrics])

### 5. Gradient Boosting – Hyperparameter Tuning + SMOTE

In [ ]:
gb_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gb_params,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

gb_grid.fit(X_train_smote, y_train_smote)
print("Best parameters:", gb_grid.best_params_)

gb_metrics, gb_fitted = evaluate_classifier(
    gb_grid.best_estimator_, X_train_smote, y_train_smote, X_test, y_test, "Gradient Boosting (Tuned + SMOTE)"
)
pd.DataFrame([gb_metrics])

### 6. XGBoost – Hyperparameter Tuning + SMOTE

In [ ]:
xgb_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 5]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, eval_metric="logloss"),
    param_grid=xgb_params,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

xgb_grid.fit(X_train_smote, y_train_smote)
print("Best parameters:", xgb_grid.best_params_)

xgb_metrics, xgb_fitted = evaluate_classifier(
    xgb_grid.best_estimator_, X_train_smote, y_train_smote, X_test, y_test, "XGBoost (Tuned + SMOTE)"
)
pd.DataFrame([xgb_metrics])

### 7. Stacking – SMOTE

In [ ]:
estimators = [
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)),
    ("xgb", XGBClassifier(n_estimators=100, max_depth=4, random_state=42, verbosity=0, eval_metric="logloss"))
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=GradientBoostingClassifier(n_estimators=50, random_state=42),
    n_jobs=-1
)

stack_metrics, stack_fitted = evaluate_classifier(
    stack_model, X_train_smote, y_train_smote, X_test, y_test, "Stacking (SMOTE)"
)
pd.DataFrame([stack_metrics])

## Model Comparison

In [ ]:
all_metrics = [dt_metrics, bag_metrics, rf_metrics, ada_metrics, gb_metrics, xgb_metrics, stack_metrics]
comparison_df = pd.DataFrame(all_metrics).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x="F1-Score", y="Model", data=comparison_df, ax=ax[0], palette="viridis")
ax[0].set_title("Test F1-Score by Model")
ax[0].set_xlim(0, 1)

sns.barplot(x="ROC-AUC", y="Model", data=comparison_df, ax=ax[1], palette="magma")
ax[1].set_title("Test ROC-AUC by Model")
ax[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

### Confusion Matrix & Classification Report (Best Model)

In [ ]:
best_model = rf_fitted
y_pred_best = best_model.predict(X_test)

print("Classification Report – Random Forest (Tuned + SMOTE)")
print(classification_report(y_test, y_pred_best, target_names=["Non-Premium", "Premium"]))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Premium", "Premium"],
            yticklabels=["Non-Premium", "Premium"])
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()

## Feature Importance

In [ ]:
importances = pd.Series(rf_fitted.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette="coolwarm")
plt.title("Feature Importance – Random Forest")
plt.tight_layout()
plt.show()

print(importances)

## Business Recommendations

1. **Prioritise Alcohol Content** – Alcohol is the strongest predictor of premium quality. Maintain processes that achieve higher alcohol levels.

2. **Control Volatile Acidity** – Keep acetic-acid levels low through better hygiene and oxidation control.

3. **Leverage Sulphates & Citric Acid** – Higher levels of these are associated with better scores.

4. **Deploy the Model** – Use the best-performing tuned ensemble (trained with SMOTE) to flag premium batches for faster review.

5. **Address Class Imbalance** – Because premium wines are rare, the model was trained with SMOTE to improve minority-class recall.

6. **Continuous Refresh** – Retrain periodically with new production data.

## Conclusion

This case study delivered an end-to-end solution for identifying premium wines.

Key takeaways:
- Alcohol, volatile acidity, sulphates and citric acid are the strongest signals.
- Class imbalance was handled with simple SMOTE (and RandomUnderSampler).
- Tuned ensemble models achieve solid F1-scores and ROC-AUC.
- F1-score and ROC-AUC are more informative than accuracy alone for this imbalanced problem.

Deploying the selected model can reduce reliance on costly sensory panels and help Moonshine allocate quality-control resources more efficiently.